# OntologySim V2 State Audit — MEDIUM Generation (Landing Layer)

Landing-layer audit for generation `8573dcf32f2ed486` (profile: medium, schema_version: 3).
Validates manifest metadata, file presence, file sizes, source family coverage, and the large ConMod runtime file.

In [0]:
import json
from pyspark.sql import functions as F

GENERATION_ID = "8573dcf32f2ed486"
PROFILE = "medium"
SCHEMA_VERSION = 3
LANDING_BASE = f"/Volumes/ontologysim/landing/source_files/{PROFILE}/{GENERATION_ID}"

# ── Load manifest ──
manifest_path = f"{LANDING_BASE}/manifest.json"
manifest = json.loads(dbutils.fs.head(manifest_path, 500_000))

# ── Validate manifest metadata ──
gen_id_ok = manifest.get("generation_id") == GENERATION_ID
profile_ok = manifest.get("profile") == PROFILE
schema_ok = manifest.get("schema_version") == SCHEMA_VERSION

print("=" * 60)
print("MANIFEST METADATA VALIDATION")
print("=" * 60)
print(f"  generation_id:   {manifest.get('generation_id')}   {'PASS' if gen_id_ok else 'FAIL — expected ' + GENERATION_ID}")
print(f"  profile:         {manifest.get('profile')}   {'PASS' if profile_ok else 'FAIL — expected ' + PROFILE}")
print(f"  schema_version:  {manifest.get('schema_version')}   {'PASS' if schema_ok else 'FAIL — expected ' + str(SCHEMA_VERSION)}")
print(f"  generated_at:    {manifest.get('generated_at')}")
print(f"  scenario:        {manifest.get('scenario')}")
print(f"  seed:            {manifest.get('seed')}")

print("\nsource_counts (exactly as stored in manifest):")
print(json.dumps(manifest.get("source_counts"), indent=2))

manifest_files = manifest.get("files", [])
print(f"\nManifest total_size_bytes: {manifest.get('total_size_bytes')}")
print(f"Manifest declared file count: {len(manifest_files)}")

MANIFEST METADATA VALIDATION
  generation_id:   8573dcf32f2ed486   PASS
  profile:         medium   PASS
  schema_version:  3   PASS
  generated_at:    2026-10-12T14:46:31.117384+00:00
  scenario:        multi_scenario
  seed:            20261001

source_counts (exactly as stored in manifest):
{
  "requirements": 8040,
  "components": 1600,
  "test_management": 16080,
  "test_execution": 160800,
  "hcp3_logs": 10665216,
  "hcp5_logs": 11429471,
  "conmod_logs": 16497867,
  "defects": 3216,
  "traces": 155976,
  "measurements": 7603360
}

Manifest total_size_bytes: 12689236373
Manifest declared file count: 10


In [0]:
# ── Compare every physical file against the manifest ──

expected_sizes = {f["path"]: f["size_bytes"] for f in manifest_files}

def list_recursive(path):
    rows = []
    for item in dbutils.fs.ls(path):
        if item.isDir():
            rows.extend(list_recursive(item.path))
        else:
            rows.append((item.path, item.size))
    return rows

comparison_rows = []
physical_rel_paths = set()

for abs_path, phys_size in list_recursive(LANDING_BASE):
    rel_path = abs_path.replace(LANDING_BASE + "/", "")
    if rel_path.startswith("dbfs:"):
        rel_path = rel_path[5:]
    if rel_path == "manifest.json":
        continue
    physical_rel_paths.add(rel_path)
    exp_size = expected_sizes.get(rel_path)
    if exp_size is None:
        status = "UNEXPECTED (not in manifest)"
    elif phys_size != exp_size:
        status = "SIZE MISMATCH"
    else:
        status = "OK"
    comparison_rows.append((rel_path, phys_size, exp_size, status))

# Check for missing files (in manifest but not on disk)
for rel_path, exp_size in expected_sizes.items():
    if rel_path not in physical_rel_paths:
        comparison_rows.append((rel_path, None, exp_size, "MISSING (not on disk)"))

comparison_df = spark.createDataFrame(
    comparison_rows,
    ["relative_path", "physical_size", "expected_size", "status"],
)

display(comparison_df.orderBy("relative_path"))

relative_path,physical_size,expected_size,status
architecture/architecture.arxml,910624,910624,OK
defects/kpm_defects.xlsx,293119,293119,OK
measurements/vehicle_signals.parquet,56739234,56739234,OK
requirements/requirements.reqif,7110261,7110261,OK
runtime/conmod/conmod_runtime.jsonl,7393290786,7393290786,OK
runtime/hcp3/hcp3_runtime.dlt,2302705132,2302705132,OK
runtime/hcp5/hcp5_runtime.log,2776212732,2776212732,OK
test_execution/test_executions.jsonl,118069296,118069296,OK
test_management/test_cases.csv,7204776,7204776,OK
traces/trace_export.tsv,26700413,26700413,OK


In [0]:
# ── Verify all 10 source families ──

expected_families = {
    "requirements":            "requirements/requirements.reqif",
    "architecture/components":  "architecture/architecture.arxml",
    "test_management":         "test_management/test_cases.csv",
    "test_execution":          "test_execution/test_executions.jsonl",
    "hcp3 runtime":             "runtime/hcp3/hcp3_runtime.dlt",
    "hcp5 runtime":             "runtime/hcp5/hcp5_runtime.log",
    "conmod runtime":           "runtime/conmod/conmod_runtime.jsonl",
    "defects":                  "defects/kpm_defects.xlsx",
    "traces":                   "traces/trace_export.tsv",
    "measurements":             "measurements/vehicle_signals.parquet",
}

manifest_paths = {f["path"] for f in manifest_files}

family_rows = []
for family, expected_file in expected_families.items():
    in_manifest = expected_file in manifest_paths
    on_disk = expected_file in physical_rel_paths
    status = "OK" if (in_manifest and on_disk) else "FAIL"
    family_rows.append((family, expected_file, in_manifest, on_disk, status))

family_df = spark.createDataFrame(
    family_rows,
    ["source_family", "expected_file", "in_manifest", "on_disk", "status"],
)

all_families_ok = all(r[4] == "OK" for r in family_rows)

print(f"Source families: {len(family_rows)}")
print(f"All families present: {'PASS' if all_families_ok else 'FAIL'}\n")

display(family_df.orderBy("source_family"))

Source families: 10
All families present: PASS



source_family,expected_file,in_manifest,on_disk,status
architecture/components,architecture/architecture.arxml,true,true,OK
conmod runtime,runtime/conmod/conmod_runtime.jsonl,true,true,OK
defects,defects/kpm_defects.xlsx,true,true,OK
hcp3 runtime,runtime/hcp3/hcp3_runtime.dlt,true,true,OK
hcp5 runtime,runtime/hcp5/hcp5_runtime.log,true,true,OK
measurements,measurements/vehicle_signals.parquet,true,true,OK
requirements,requirements/requirements.reqif,true,true,OK
test_execution,test_execution/test_executions.jsonl,true,true,OK
test_management,test_management/test_cases.csv,true,true,OK
traces,traces/trace_export.tsv,true,true,OK


In [0]:
# ── Specifically verify the large ConMod file ──
conmod_rel = "runtime/conmod/conmod_runtime.jsonl"
conmod_manifest = next((f for f in manifest_files if f["path"] == conmod_rel), None)
conmod_phys = next((r for r in comparison_rows if r[0] == conmod_rel), None)

print("=" * 60)
print("CONMOD LARGE FILE VERIFICATION")
print("=" * 60)

conmod_ok = False
if conmod_manifest:
    print(f"  Manifest path:       {conmod_manifest['path']}")
    print(f"  Manifest size_bytes: {conmod_manifest['size_bytes']:,}")
else:
    print("  FAIL: ConMod file NOT found in manifest")

if conmod_phys:
    rel_path, phys_size, exp_size, status = conmod_phys
    print(f"  Physical size_bytes: {phys_size:,}")
    print(f"  Size (human):        {phys_size / (1024**3):.2f} GB")
    if conmod_manifest:
        conmod_ok = phys_size == conmod_manifest["size_bytes"]
        print(f"  Size match:          {'PASS' if conmod_ok else 'FAIL'}")
else:
    print("  FAIL: ConMod file NOT found on disk")

# ── Final PASS/FAIL report ──
print("\n" + "=" * 60)
print("OVERALL LANDING AUDIT REPORT")
print("=" * 60)

missing_files = [r[0] for r in comparison_rows if "MISSING" in r[3]]
size_mismatches = [r[0] for r in comparison_rows if "SIZE MISMATCH" in r[3]]
unexpected_files = [r[0] for r in comparison_rows if "UNEXPECTED" in r[3]]

checks = {
    "Manifest generation_id match":   gen_id_ok,
    "Manifest profile match":         profile_ok,
    "Manifest schema_version match":   schema_ok,
    "No missing files":               len(missing_files) == 0,
    "No size mismatches":             len(size_mismatches) == 0,
    "No unexpected files":            len(unexpected_files) == 0,
    "All 10 source families present": all_families_ok,
    "ConMod large file verified":      conmod_ok,
}

for name, passed in checks.items():
    print(f"  {'PASS' if passed else 'FAIL'}  {name}")

if missing_files:
    print(f"\n  Missing files: {missing_files}")
if size_mismatches:
    print(f"  Size mismatches: {size_mismatches}")
if unexpected_files:
    print(f"  Unexpected files: {unexpected_files}")

all_pass = all(checks.values())
print(f"\n  {'=' * 40}")
print(f"  OVERALL LANDING: {'PASS' if all_pass else 'FAIL'}")
print(f"  {'=' * 40}")

CONMOD LARGE FILE VERIFICATION
  Manifest path:       runtime/conmod/conmod_runtime.jsonl
  Manifest size_bytes: 7,393,290,786
  Physical size_bytes: 7,393,290,786
  Size (human):        6.89 GB
  Size match:          PASS

OVERALL LANDING AUDIT REPORT
  PASS  Manifest generation_id match
  PASS  Manifest profile match
  PASS  Manifest schema_version match
  PASS  No missing files
  PASS  No size mismatches
  PASS  No unexpected files
  PASS  All 10 source families present
  PASS  ConMod large file verified

  OVERALL LANDING: PASS
